# Thảo luận và Minh hoạ quá trình Làm sạch Dữ liệu (Data Cleaning)

Notebook này minh hoạ trạng thái của dữ liệu **Trước** và **Sau** khi làm sạch để chúng ta có thể dễ dàng thảo luận.

## 1. Dữ liệu Trước khi làm sạch (Before Cleaning)

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Đọc dữ liệu
df = pd.read_csv('data/train.csv')
print("Kích thước tập dữ liệu ban đầu:", df.shape)
df.head()

Kích thước tập dữ liệu ban đầu: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [2]:
# Kiểm tra tỷ lệ dữ liệu bị thiếu ở các cột
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({'Missing Values': missing_data, 'Percentage': missing_percentage})
missing_df = missing_df[missing_df['Missing Values'] > 0].sort_values(by='Percentage', ascending=False)

print("Các cột có dữ liệu bị thiếu:")
missing_df

Các cột có dữ liệu bị thiếu:


,Missing Values,Percentage
PoolQC,1453,99.520548
MiscFeature,1406,96.301370
Alley,1369,93.767123
Fence,1179,80.753425
MasVnrType,872,59.726027
FireplaceQu,690,47.260274
LotFrontage,259,17.739726
GarageType,81,5.547945
GarageYrBlt,81,5.547945
GarageFinish,81,5.547945


## 2. Các bước Làm sạch Dữ liệu (Cleaning Strategy)

Dựa trên quan sát ban đầu, chúng ta có thể áp dụng các bước sau:
1. **Loại bỏ các cột thiếu quá nhiều dữ liệu** (ví dụ: > 80% như PoolQC, MiscFeature, Alley, Fence).
2. **Điền giá trị thiếu (Imputation)**:
    - Với dữ liệu hạng mục (Categorical - ví dụ GarageType, BsmtQual): Điền bằng 'None' (đại diện cho việc không có Garage hoặc không có Basement).
    - Với dữ liệu số (Numerical - ví dụ LotFrontage): Điền bằng giá trị trung vị (Median).

In [3]:
df_cleaned = df.copy()

# 1. Bỏ các cột thiếu trên 80% dữ liệu
cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']
df_cleaned.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# 2. Xử lý giá trị thiếu cho các biến phân loại (Categorical)
cat_cols = df_cleaned.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df_cleaned[col].isnull().any():
        # Nếu nhà không có các thuộc tính này thì điền là 'None'
        df_cleaned[col] = df_cleaned[col].fillna('None')

# 3. Xử lý giá trị thiếu cho các biến số (Numerical)
num_cols = df_cleaned.select_dtypes(exclude=['object']).columns
for col in num_cols:
    if df_cleaned[col].isnull().any():
        # Điền bằng trung vị
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

## 3. Dữ liệu Sau khi làm sạch (After Cleaning)

In [4]:
print("Kích thước tập dữ liệu sau khi làm sạch:", df_cleaned.shape)
print("Tổng số lượng giá trị thiếu còn lại trong toàn bộ dataset:", df_cleaned.isnull().sum().sum())

Kích thước tập dữ liệu sau khi làm sạch: (1460, 77)
Tổng số lượng giá trị thiếu còn lại trong toàn bộ dataset: 0


In [5]:
# Xem qua một vài dòng sau khi làm sạch
df_cleaned.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,Reg,Lvl,AllPub,FR2,...,0,0,0,0,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0,0,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,IR1,Lvl,AllPub,Corner,...,272,0,0,0,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,IR1,Lvl,AllPub,FR2,...,0,0,0,0,0,12,2008,WD,Normal,250000
